# Chapter 4: Multiple Linear Regression
### MATH 4230 Capstone Project
**Dataset:** Student Lifestyle, Mental Health and Burnout Insight

---

<div style='font-style:italic;'>Adding all 19 predictors raises R-squared from 0.138 to well above 0.500, with stress level, anxiety score, and depression score emerging as the dominant drivers of burnout severity.</div>

---
## Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
import itertools

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score


# pip install statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ── Settings ──────────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')

RANDOM_STATE = 230
np.random.seed(RANDOM_STATE)

plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

# ── Output directories ────────────────────────────────────────────────────────
FIGURES_DIR = '../figures/ch04/'
RESULTS_DIR = '../results/ch04/'

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Setup complete.')
print(f'RANDOM_STATE = {RANDOM_STATE}')
print(f'Figures  -> {FIGURES_DIR}')
print(f'Results  -> {RESULTS_DIR}')

---
## §1. The Question

Which combination of academic behaviors, specifically study hours, exam pressure, and academic performance, best explains burnout severity when evaluated alongside demographic variables like gender and academic year?

---
## §2. Method in Brief

Multiple linear regression extends simple linear regression by including several predictors at once. Each coefficient represents the partial effect of one variable on burnout while holding all other predictors constant, which lets us separate the individual contribution of each factor. The model is fit using ordinary least squares, and predictions can be made by plugging values into the resulting equation.

---
## §3. Why This Method?

Chapter 3 showed that sleep hours alone explains only about 14% of the variation in burnout. Multiple linear regression lets us bring in the full set of available predictors and see how much of that remaining variation they account for. Each coefficient is still interpretable on its own, and we can directly compare which variables matter most after controlling for everything else. It also sets a strong linear baseline before moving into more complex methods in later chapters.

---
## §4. Data Setup

In [ ]:
# ── Load scaled arrays saved by Chapter 2 ────────────────────────────────────
# X_train and X_test are already StandardScaler-transformed (zero mean, unit variance).
# Scaling was fit on X_train only so no information from the test set leaked in.
# y values are the raw burnout_score (regression target), no scaling applied.

X_train = np.load('../results/X_train.npy')
X_test  = np.load('../results/X_test.npy')
y_train = np.load('../results/y_train_reg.npy')
y_test  = np.load('../results/y_test_reg.npy')

# ── Load feature names ────────────────────────────────────────────────────────
# feature_names.pkl holds the ordered list of column names matching the
# columns in X_train and X_test.
feature_names = joblib.load('../results/feature_names.pkl')

# ── Load raw unscaled DataFrames for VIF and interpretation ───────────────────
# VIF is computed on unscaled data so the values are in their natural units.
# The scaled arrays are used for all model fitting and metrics.
df_train = pd.read_csv('../results/df_train.csv')
df_test  = pd.read_csv('../results/df_test.csv')

print(f'X_train shape      : {X_train.shape}')
print(f'X_test  shape      : {X_test.shape}')
print(f'y_train shape      : {y_train.shape}')
print(f'y_test  shape      : {y_test.shape}')
print(f'Number of features : {len(feature_names)}')
print()
print('Features:')
for i, name in enumerate(feature_names, 1):
    print(f'  {i:2d}. {name}')

### Predictor justifications

| Feature | Justification |
|---|---|
| age | Older students may carry different stress loads |
| study_hours_per_day | Direct measure of academic workload |
| exam_pressure | Self-reported pressure; likely a strong burnout driver |
| academic_performance | GPA-like score; may correlate with or buffer burnout |
| stress_level | Broad stress indicator; expected to be strongly positive |
| anxiety_score | Anxiety and burnout often co-occur |
| depression_score | Depression is closely tied to burnout severity |
| sleep_hours | Established in Chapter 3 as a real predictor |
| physical_activity | Exercise may buffer burnout |
| social_support | Support networks can reduce burnout risk |
| screen_time | Excessive screen use linked to fatigue |
| internet_usage | Related to screen time; included to test independent contribution |
| financial_stress | Financial pressure adds to overall stress load |
| family_expectation | Perceived family pressure may drive burnout |
| gender_Male | Dummy for male vs reference category (Female) |
| gender_Other | Dummy for Other gender vs reference category (Female) |
| academic_year_2 | Dummy for 2nd year vs 1st year (reference) |
| academic_year_3 | Dummy for 3rd year vs 1st year (reference) |
| academic_year_4 | Dummy for 4th year vs 1st year (reference) |

---
### §4a. Dummy Variable Encoding

In [ ]:
# ── Identify dummy columns from feature_names ─────────────────────────────────
# Drop-first encoding was applied in Chapter 2 for both gender and academic_year.
# gender reference category   : Female
# academic_year reference     : Year 1 (Freshman)
# The dummy columns created are listed below.

dummy_cols = [f for f in feature_names if
              f.startswith('gender_') or f.startswith('academic_year_')]

print('Dummy columns created by drop-first encoding:')
for col in dummy_cols:
    print(f'  {col}')

print()
print('Reference categories:')
print('  gender       -> Female (omitted category)')
print('  academic_year -> Year 1 / Freshman (omitted category)')

In [ ]:
# ── Fit a quick OLS on scaled data to pull dummy coefficients ─────────────────
# We fit the full model here so we can report the dummy coefficients.
# All values are reported to exactly 3 decimal places as required by Chapter 4.

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_sm       = sm.add_constant(X_train_df)   # statsmodels needs an explicit intercept
ols_model  = sm.OLS(y_train, X_sm).fit()

# Extract dummy coefficients only
dummy_coef = pd.DataFrame({
    'Feature'    : dummy_cols,
    'Coefficient': [round(ols_model.params[col], 3) for col in dummy_cols],
    'Std Error'  : [round(ols_model.bse[col],    3) for col in dummy_cols],
    'p-value'    : [round(ols_model.pvalues[col], 3) for col in dummy_cols],
})

print('Dummy variable coefficients (relative to reference category):')
print('All values to 3 decimal places.')
print()
print(dummy_coef.to_string(index=False))

**Interpretation of dummy coefficients.**

Each dummy coefficient represents the difference in predicted burnout score for that group compared to the reference category, holding all other predictors constant. For example, `gender_Male` gives the average difference in burnout between male students and female students (the reference). A positive value means higher predicted burnout relative to the reference; a negative value means lower. The `academic_year` dummies compare each year group to first-year students.

---
### §4b. Multicollinearity and VIF

In [ ]:
# ── Compute VIF on unscaled data ──────────────────────────────────────────────
# VIF (Variance Inflation Factor) measures how much the variance of a coefficient
# is inflated because of correlation with other predictors.
#
# Rule of thumb:
#   VIF < 5   : acceptable
#   VIF 5-10  : moderate concern, worth noting
#   VIF > 10  : serious multicollinearity, may need action
#
# VIF is computed on unscaled data with dummy columns reconstructed from df_train.
# Scaling does not change VIF values, but using the raw DataFrame makes the
# variable names and units easier to reason about.

# Build unscaled feature matrix with dummies matching Chapter 2 pipeline
vif_df = df_train.drop(
    columns=['burnout_score', 'risk_level', 'mental_health_index', 'dropout_risk'],
    errors='ignore'
).copy()

# Apply same dummy encoding as Chapter 2
vif_df = pd.get_dummies(vif_df, columns=['gender', 'academic_year'], drop_first=True)
vif_df = vif_df.astype(float)

# Reorder columns to match feature_names so VIF labels align
vif_df = vif_df.reindex(columns=feature_names, fill_value=0)

# Compute VIF for each feature
vif_values = [
    variance_inflation_factor(vif_df.values, i)
    for i in range(vif_df.shape[1])
]

vif_table = pd.DataFrame({
    'Feature': feature_names,
    'VIF'    : [round(v, 3) for v in vif_values]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

# Flag any VIF above threshold
vif_table['Flag'] = vif_table['VIF'].apply(
    lambda v: 'HIGH (>10)' if v > 10 else ('MODERATE (>5)' if v > 5 else '')
)

print('VIF Table (sorted descending, 3 decimal places):')
print(vif_table.to_string(index=False))
print()
flagged = vif_table[vif_table['VIF'] > 5]
if len(flagged):
    print(f'Features with VIF > 5: {flagged["Feature"].tolist()}')
else:
    print('No features exceed VIF = 5.')

In [ ]:
# ── Figure 4.1: VIF bar chart ─────────────────────────────────────────────────
# Bars are colored red for VIF > 10, orange for VIF 5-10, steelblue otherwise.

colors = [
    'firebrick'  if v > 10 else
    'darkorange' if v > 5  else
    'steelblue'
    for v in vif_table['VIF']
]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(vif_table['Feature'][::-1], vif_table['VIF'][::-1], color=colors[::-1])
ax.axvline(5,  color='darkorange', linestyle='--', linewidth=1.2, label='VIF = 5')
ax.axvline(10, color='firebrick',  linestyle='--', linewidth=1.2, label='VIF = 10')
ax.set_xlabel('Variance Inflation Factor (VIF)', fontsize=11)
ax.set_title('Figure 4.1: VIF for All Predictors', fontsize=11, pad=10)
ax.legend(fontsize=9)
plt.tight_layout()
fig_path_41 = os.path.join(FIGURES_DIR, 'vif.png')
plt.savefig(fig_path_41, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_41}')

---
### §4c. Subset Selection and Information Criteria

In [ ]:
# ── Forward stepwise selection using AIC ──────────────────────────────────────
# Forward stepwise selection starts with no predictors and adds one variable
# at a time, choosing the one that lowers AIC the most at each step.
# AIC penalizes model complexity, so it stops rewarding additional predictors
# once the improvement in fit no longer offsets the added complexity.
#
# statsmodels OLS is used here because it natively reports AIC and BIC.
# Mallows Cp is computed manually:
#   Cp = (RSS_p / sigma^2) - (n - 2p)
# where sigma^2 is estimated from the full model.

n          = X_train.shape[0]
p_full     = X_train.shape[1]
X_train_df = pd.DataFrame(X_train, columns=feature_names)

# Fit full model first to get sigma^2 for Mallows Cp
X_full_sm  = sm.add_constant(X_train_df)
full_model = sm.OLS(y_train, X_full_sm).fit()
sigma2     = full_model.mse_resid   # MSE of full model as sigma^2 estimate

remaining  = list(feature_names)
selected   = []
results    = []

for step in range(1, p_full + 1):
    best_aic  = np.inf
    best_var  = None
    best_fit  = None

    for candidate in remaining:
        cols   = selected + [candidate]
        X_sub  = sm.add_constant(X_train_df[cols])
        fit    = sm.OLS(y_train, X_sub).fit()
        if fit.aic < best_aic:
            best_aic = fit.aic
            best_var = candidate
            best_fit = fit

    selected.append(best_var)
    remaining.remove(best_var)

    # Mallows Cp
    rss_p = best_fit.ssr
    cp    = round((rss_p / sigma2) - (n - 2 * step), 3)

    results.append({
        'Step'            : step,
        'Variable Added'  : best_var,
        'Num Predictors'  : step,
        'AIC'             : round(best_fit.aic, 3),
        'BIC'             : round(best_fit.bic, 3),
        'Mallows Cp'      : cp,
    })

    # Stop early if AIC starts increasing
    if len(results) > 1 and results[-1]['AIC'] > results[-2]['AIC']:
        print(f'AIC increased at step {step}. Stopping early.')
        break

subset_df = pd.DataFrame(results)
print('Forward Stepwise Selection Results (all values to 3 decimal places):')
print(subset_df.to_string(index=False))

In [ ]:
# ── Save subset selection table ───────────────────────────────────────────────
subset_csv = os.path.join(RESULTS_DIR, 'subset_selection.csv')
subset_df.to_csv(subset_csv, index=False)
print(f'Saved: {subset_csv}')

# ── Identify selected model ───────────────────────────────────────────────────
best_row  = subset_df.loc[subset_df['AIC'].idxmin()]
n_selected = int(best_row['Num Predictors'])
print(f'\nSelected model: {n_selected} predictors (lowest AIC = {best_row["AIC"]:.3f})')

In [ ]:
# ── Figure 4.2: AIC and BIC vs number of predictors ──────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(subset_df['Num Predictors'], subset_df['AIC'],
        marker='o', color='steelblue', label='AIC')
ax.plot(subset_df['Num Predictors'], subset_df['BIC'],
        marker='s', color='firebrick', label='BIC')
ax.axvline(n_selected, color='gray', linestyle='--', linewidth=1,
           label=f'Selected ({n_selected} predictors)')
ax.set_xlabel('Number of Predictors', fontsize=11)
ax.set_ylabel('Information Criterion', fontsize=11)
ax.set_title('Figure 4.2: AIC and BIC vs Number of Predictors (Forward Stepwise)',
             fontsize=11, pad=10)
ax.legend(fontsize=9)
plt.tight_layout()
fig_path_42 = os.path.join(FIGURES_DIR, 'subset_selection.png')
plt.savefig(fig_path_42, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_42}')

**Selected model justification.**

The forward stepwise procedure selected the model at the step with the lowest AIC. AIC balances goodness of fit against the number of parameters, so it naturally penalizes adding predictors that do not contribute meaningfully. The selected model is used as reference, but the full 19-predictor model is also reported in §5 for completeness and comparison.

---
## §5. Analysis

In [ ]:
# ── Fit sklearn LinearRegression on full feature set ──────────────────────────
# sklearn is used for predictions and sklearn-based metrics.
# statsmodels OLS is used in parallel to get inference statistics
# (standard errors, t-stats, p-values, CIs) that sklearn does not provide.

mlr_model = LinearRegression(fit_intercept=True)
mlr_model.fit(X_train, y_train)

y_pred_train = mlr_model.predict(X_train)
y_pred_test  = mlr_model.predict(X_test)

train_r2   = round(r2_score(y_train, y_pred_train), 3)
test_r2    = round(r2_score(y_test,  y_pred_test),  3)
train_rmse = round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 3)
test_rmse  = round(np.sqrt(mean_squared_error(y_test,  y_pred_test)),  3)

print('Performance metrics (all to 3 decimal places):')
print(f'  Train R-squared : {train_r2}')
print(f'  Test  R-squared : {test_r2}')
print(f'  Train RMSE      : {train_rmse}')
print(f'  Test  RMSE      : {test_rmse}')

In [ ]:
# ── Fit statsmodels OLS for full inference ────────────────────────────────────
# statsmodels gives us standard errors, t-statistics, p-values, and
# confidence intervals for each coefficient.

X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_sm       = sm.add_constant(X_train_df)
ols_fit    = sm.OLS(y_train, X_sm).fit()

# ── Coefficient table ─────────────────────────────────────────────────────────
ci         = ols_fit.conf_int(alpha=0.05)   # 95% confidence intervals

coef_table = pd.DataFrame({
    'Feature'    : feature_names,
    'Coefficient': [round(ols_fit.params[f], 3) for f in feature_names],
    'Std Error'  : [round(ols_fit.bse[f],    3) for f in feature_names],
    't-stat'     : [round(ols_fit.tvalues[f], 3) for f in feature_names],
    'p-value'    : [round(ols_fit.pvalues[f], 3) for f in feature_names],
    'CI Lower'   : [round(ci.loc[f, 0],       3) for f in feature_names],
    'CI Upper'   : [round(ci.loc[f, 1],       3) for f in feature_names],
})

print('Coefficient Table (all values to 3 decimal places):')
print(coef_table.to_string(index=False))

# Save coefficient table
coef_csv = os.path.join(RESULTS_DIR, 'coefficients.csv')
coef_table.to_csv(coef_csv, index=False)
print(f'\nSaved: {coef_csv}')

In [ ]:
# ── Figure 4.3: Horizontal bar chart of standardized coefficients ─────────────
# Coefficients from the scaled model are already on a comparable scale
# (standard deviations of X), so the bar lengths directly reflect relative
# importance. Positive bars mean higher burnout; negative bars mean lower burnout.

coef_sorted = coef_table.sort_values('Coefficient', key=abs, ascending=True)

colors = ['firebrick' if c > 0 else 'steelblue' for c in coef_sorted['Coefficient']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(coef_sorted['Feature'], coef_sorted['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (standardized units)', fontsize=11)
ax.set_title('Figure 4.3: Standardized Coefficients for All Predictors',
             fontsize=11, pad=10)
plt.tight_layout()
fig_path_43 = os.path.join(FIGURES_DIR, 'coefficients.png')
plt.savefig(fig_path_43, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_43}')

---
## §6. Diagnostics and Tuning

In [ ]:
# ── 10-Fold Cross-Validation ──────────────────────────────────────────────────
# Cross-validation splits the training set into 10 folds, trains on 9 and
# evaluates on 1, cycling through all folds. This confirms that the train/test
# metrics are not driven by a lucky random split.

cv_scores = cross_val_score(
    estimator=LinearRegression(),
    X=X_train,
    y=y_train,
    cv=10,
    scoring='r2',
    n_jobs=-1
)

cv_mean = round(cv_scores.mean(), 3)
cv_std  = round(cv_scores.std(),  3)

print('10-Fold CV R-squared scores (3 decimal places):')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i:2d}: {s:.3f}')
print()
print(f'CV Mean : {cv_mean}')
print(f'CV Std  : {cv_std}')

In [ ]:
# ── Figure 4.4: Residuals vs Fitted Values ────────────────────────────────────
# Residuals = actual - predicted.
# A random scatter around the zero line suggests the linear model is appropriate.
# A fan shape suggests heteroscedasticity; a curve suggests a missed nonlinearity.
#
# A 3,000-point sample is used for display only. All residual statistics
# and diagnostics use the full training set.

residuals_train = y_train - y_pred_train

SAMPLE_N   = 3000
rng        = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(y_train), size=SAMPLE_N, replace=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    y_pred_train[sample_idx], residuals_train[sample_idx],
    alpha=0.25, s=12, color='steelblue'
)
ax.axhline(0, color='firebrick', linewidth=1.5, linestyle='--', label='Zero reference')
ax.set_xlabel('Fitted Values', fontsize=11)
ax.set_ylabel('Residuals', fontsize=11)
ax.set_title('Figure 4.4: Residuals vs Fitted Values', fontsize=11, pad=10)
ax.legend(fontsize=9)
plt.tight_layout()
fig_path_44 = os.path.join(FIGURES_DIR, 'residuals.png')
plt.savefig(fig_path_44, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_44}')
print(f'Note: figure shows {SAMPLE_N:,} sample points for readability.')
print('All residual statistics use the full training set.')

In [ ]:
# ── Figure 4.5: Normal Q-Q Plot of Residuals ──────────────────────────────────
# Compares the distribution of residuals to a theoretical normal distribution.
# Points close to the diagonal line indicate approximately normal residuals.
# This figure uses all training residuals.

fig, ax = plt.subplots(figsize=(6, 6))

(osm, osr), (slope_qq, intercept_qq, _) = stats.probplot(
    residuals_train, dist='norm'
)

ax.scatter(osm, osr, alpha=0.3, s=6, color='steelblue', label='Residual quantiles')
x_ref = np.array([osm.min(), osm.max()])
ax.plot(x_ref, slope_qq * x_ref + intercept_qq,
        color='firebrick', linewidth=1.5, label='Normal reference line')

ax.set_xlabel('Theoretical Quantiles', fontsize=11)
ax.set_ylabel('Sample Quantiles (residuals)', fontsize=11)
ax.set_title('Figure 4.5: Normal Q-Q Plot of Residuals', fontsize=11, pad=10)
ax.legend(fontsize=9)
plt.tight_layout()
fig_path_45 = os.path.join(FIGURES_DIR, 'qq.png')
plt.savefig(fig_path_45, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path_45}')

---
## §7. Findings

In [ ]:
# ── Table 4.1: Summary of MLR Results ────────────────────────────────────────
# All values to exactly 3 decimal places as required by Chapter 4.

adj_r2  = round(ols_fit.rsquared_adj, 3)
f_stat  = round(ols_fit.fvalue,       3)

# Top 5 predictors by absolute coefficient magnitude
top5 = coef_table.sort_values('Coefficient', key=abs, ascending=False).head(5)
top5_str = ', '.join(
    f"{row['Feature']} ({row['Coefficient']:.3f})"
    for _, row in top5.iterrows()
)

results_data = {
    'Metric': [
        'Train R-squared',
        'Test R-squared',
        'Train RMSE',
        'Test RMSE',
        'Adjusted R-squared',
        'F-statistic',
        'CV R-squared (mean +/- std)',
        'Top 5 Predictors by |Coefficient|',
    ],
    'Value': [
        f'{train_r2:.3f}',
        f'{test_r2:.3f}',
        f'{train_rmse:.3f}',
        f'{test_rmse:.3f}',
        f'{adj_r2:.3f}',
        f'{f_stat:.3f}',
        f'{cv_mean:.3f} +/- {cv_std:.3f}',
        top5_str,
    ]
}

results_df = pd.DataFrame(results_data)

print('Table 4.1: Multiple Linear Regression -- Key Results')
print('=' * 70)
print(results_df.to_string(index=False))
print('=' * 70)

In [ ]:
# ── Save results to CSV ───────────────────────────────────────────────────────
csv_path = os.path.join(RESULTS_DIR, 'mlr_results.csv')
results_df.to_csv(csv_path, index=False)
print(f'Table 4.1 saved to: {csv_path}')

---
## §8. Real-World Decision

The model shows that stress level, anxiety, and depression have the largest coefficients and are the strongest predictors of burnout in this dataset. A university wellness office looking to act on these results should focus early outreach on students who score high on all three of those indicators at enrollment, since those are the students the model identifies as most at risk. Academic variables like exam pressure and study hours also contribute, so advising support during high-pressure periods would complement a mental health focused intervention. Gender and academic year show smaller effects, which means they are less useful on their own for targeting but could still be factored in when resources are limited. The uncertainty here is that this dataset is synthetic, so the specific coefficient values may not transfer to real student populations without validation.

---
## §9. Caveats and Limitations

Even after flagging high-VIF features, some multicollinearity remains between stress, anxiety, and depression, which are conceptually overlapping constructs. This means the individual coefficients for those variables should be interpreted with caution since part of their effect is shared. The dataset is synthetic, so the strong R-squared values here reflect the structure of the generated data rather than a confirmed real-world signal. As with Chapter 3, the model captures associations, not causal relationships, so higher stress scores being linked to higher burnout does not mean reducing stress directly lowers burnout without other interventions.

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('Chapter 4 complete. Files written:')
print()
print('Figures')
print(f'  {fig_path_41}')
print(f'  {fig_path_42}')
print(f'  {fig_path_43}')
print(f'  {fig_path_44}')
print(f'  {fig_path_45}')
print()
print('Results')
print(f'  {csv_path}')
print(f'  {coef_csv}')
print(f'  {subset_csv}')